# 02 - Transformación y limpieza de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook toma el fichero crudo descargado en el paso anterior y produce un dataset limpio, tipado y listo para cargar en Snowflake.

**Entrada:** `data/raw/contratos_menores.xlsx`  
**Salida:** `data/processed/contratos_limpios.csv`

---

### Decisiones y supuestos documentados

| # | Decisión | Justificación |
|---|---|---|
| 1 | Se eliminan `Unnamed: 0` y `Lista de lotes` | 100 % y 99.8 % nulas respectivamente. Sin valor analítico. |
| 2 | Se separa `Contratistas` en `nif_contratista` y `nombre_contratista` | El campo mezcla CIF/NIF con nombre en un solo string (`"B27188317 - EXCAVACIONES J.CARREIRA"`). |
| 3 | Los importes (strings con formato español `"1.234,56"`) se convierten a `float` | Requiere quitar los puntos de miles y sustituir la coma decimal por punto. |
| 4 | Los nombres de columna se normalizan a snake_case sin tildes ni espacios | Facilita el trabajo en SQL y Python. |
| 5 | `Fecha formalización` (96 % nula) se conserva y se añade `fecha_estimada` sintética | El número de referencia sigue el patrón `CT NNN/AAAA`, por lo que el año es recuperable. La fecha sintética usa semilla fija para ser reproducible. |
| 6 | Se eliminan los registros `Contrato Mayor` | El proyecto es específicamente de contratos menores; los contratos mayores son ruido. |
| 7 | Se eliminan columnas sin valor analítico tras el filtro | `codigo_cpv` (100 % nulos), `entidad_contratante`, `tipo_contratacion`, `tipo_procedimiento`, `sistema_contratacion` y `tramitacion` son constantes y no discriminan nada. |

## 0 — Importaciones y rutas

In [17]:
import re           # expresiones regulares — para extraer el año del número de referencia
import numpy as np  # generación de fechas sintéticas con semilla fija
import pandas as pd
from pathlib import Path  # manejo de rutas independiente del sistema operativo

# Rutas de entrada y salida del proceso de transformación
RUTA_RAW       = Path("../data/raw/contratos_menores.xlsx")
RUTA_PROCESADO = Path("../data/processed/contratos_limpios.csv")

print(f"Leyendo: {RUTA_RAW}")

Leyendo: ../data/raw/contratos_menores.xlsx


## 1 — Cargar el fichero crudo

In [18]:
# dtype=str fuerza que todos los campos se lean como texto puro,
# evitando que pandas convierta CIFs como "07123456A" en NaN o los trunque
df = pd.read_excel(RUTA_RAW, dtype=str)

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(3)

Filas: 631  |  Columnas: 20


,Unnamed: 0,Entidad Contratante,Número de Referencia del Contrato,Objeto del Contrato,Valor estimado del contrato (en euros),Presupuesto base sin impuestos (en euros),Plazo ejecución (en meses),Código CPV del objeto del contrato,Tipo de contratación,Tipo de Contrato,Tipo de procedimiento,Sistema de contratación,Tramitación,Fecha formalización,Importe total ofertado (sin impuestos) (en euros),Importe total ofertado (con impuestos) (en euros),URL a la licitación específica del expediente,Observaciones,Contratistas,Lista de lotes
0,NaN,Ayuntamiento Vilagarcía de Arousa,CT 80/2022-e,Demolicion edificacion Avda da Mariña 15 (Coma...,"206.356,56","206.356,56",2,45111000,Contrato Mayor,Obras,Otros,No aplica,Ordinaria,09-01-2023,"123.401,22","149.315,48",https://contrataciondelestado.es/wps/poc?uri=d...,NaN,"B27188317 - EXCAVACIONES J.CARREIRA, S.L.",NaN
1,NaN,Ayuntamiento Vilagarcía de Arousa,CT 02/2023,"Suministro de papel para fotocopiadora, 600 pa...","2.394,00","2.394,00",0.07,NaN,Contrato Menor,Suministros,Contrato menor,No aplica,Ordinaria,NaN,"2.394,00","2.896,74",https://contrataciondelestado.es/wps/poc?uri=d...,NaN,B36127405 - REDINOR S.L.,NaN
2,NaN,Ayuntamiento Vilagarcía de Arousa,CT 75/2022,"Redacción de Proyecto de ""Recuperación de la F...","14.950,00","14.849,00",2,NaN,Contrato Menor,Servicios,Contrato menor,No aplica,Ordinaria,NaN,"14.849,00","17.967,29",https://contrataciondelestado.es/wps/poc?uri=d...,NaN,35458826W - MANUEL TANOIRA CARBALLO,NaN


## 2 — Eliminar columnas sin valor

In [19]:
# Columnas con 100% y 99.8% de nulos — no aportan información útil al análisis
COLUMNAS_BASURA = ["Unnamed: 0", "Lista de lotes"]
df = df.drop(columns=COLUMNAS_BASURA)

print(f"Columnas tras limpieza: {df.shape[1]}")
print(df.columns.tolist())

Columnas tras limpieza: 18
['Entidad Contratante', 'Número de Referencia del Contrato', 'Objeto del Contrato', 'Valor estimado del contrato (en euros)', 'Presupuesto base sin impuestos (en euros)', 'Plazo ejecución (en meses)', 'Código CPV del objeto del contrato', 'Tipo de contratación', 'Tipo de Contrato', 'Tipo de procedimiento', 'Sistema de contratación', 'Tramitación', 'Fecha formalización', 'Importe total ofertado (sin impuestos) (en euros)', 'Importe total ofertado (con impuestos) (en euros)', 'URL a la licitación específica del expediente', 'Observaciones', 'Contratistas']


## 3 — Renombrar columnas a snake_case

In [20]:
# Mapeamos los nombres originales del Excel (con espacios, tildes y paréntesis)
# a nombres snake_case que funcionan directamente en SQL y como atributos de Python
RENOMBRAR = {
    "Entidad Contratante":                               "entidad_contratante",
    "Número de Referencia del Contrato":                  "num_referencia",
    "Objeto del Contrato":                               "objeto_contrato",
    "Valor estimado del contrato (en euros)":             "valor_estimado_eur",
    "Presupuesto base sin impuestos (en euros)":          "presupuesto_base_eur",
    "Plazo ejecución (en meses)":                        "plazo_meses",
    "Código CPV del objeto del contrato":                 "codigo_cpv",
    "Tipo de contratación":                              "tipo_contratacion",
    "Tipo de Contrato":                                  "tipo_contrato",
    "Tipo de procedimiento":                             "tipo_procedimiento",
    "Sistema de contratación":                           "sistema_contratacion",
    "Tramitación":                                       "tramitacion",
    "Fecha formalización":                               "fecha_formalizacion",
    "Importe total ofertado (sin impuestos) (en euros)": "importe_sin_iva_eur",
    "Importe total ofertado (con impuestos) (en euros)": "importe_con_iva_eur",
    "URL a la licitación específica del expediente":      "url_licitacion",
    "Observaciones":                                     "observaciones",
    "Contratistas":                                      "contratistas_raw",
}

df = df.rename(columns=RENOMBRAR)
print(df.columns.tolist())

['entidad_contratante', 'num_referencia', 'objeto_contrato', 'valor_estimado_eur', 'presupuesto_base_eur', 'plazo_meses', 'codigo_cpv', 'tipo_contratacion', 'tipo_contrato', 'tipo_procedimiento', 'sistema_contratacion', 'tramitacion', 'fecha_formalizacion', 'importe_sin_iva_eur', 'importe_con_iva_eur', 'url_licitacion', 'observaciones', 'contratistas_raw']


## 4 — Limpiar e importar la columna de contratistas

Formato original: `"B27188317 - EXCAVACIONES J.CARREIRA, S.L."`  
Se separa por el primer ` - ` en NIF/CIF y nombre. Después se valida el formato del identificador.

In [21]:
def separar_contratista(texto):
    """Devuelve (nif, nombre) a partir de 'NIF - NOMBRE'."""
    if pd.isna(texto) or not isinstance(texto, str):
        return pd.NA, pd.NA
    # maxsplit=1 divide solo por el primer " - ", evitando romper nombres que contengan guion
    partes = texto.split(" - ", maxsplit=1)
    if len(partes) == 2:
        return partes[0].strip(), partes[1].strip()
    return pd.NA, texto.strip()

df[["nif_contratista", "nombre_contratista"]] = (
    df["contratistas_raw"]
    .apply(lambda x: pd.Series(separar_contratista(x)))
)

# ── Validación de NIF / NIE / CIF ──────────────────────────────────────────

LETRAS_NIF = "TRWAGMYFPDXBNJZSQVHLCKE"

def validar_nif(nif):
    """NIF persona física: 8 dígitos + letra. La letra = LETRAS_NIF[número % 23]."""
    return nif[8] == LETRAS_NIF[int(nif[:8]) % 23]

def validar_nie(nie):
    """NIE de extranjero: X/Y/Z + 7 dígitos + letra."""
    # Sustituimos el prefijo por su equivalente numérico antes de aplicar la fórmula NIF
    prefijo = {"X": "0", "Y": "1", "Z": "2"}
    numero  = int(prefijo[nie[0]] + nie[1:8])
    return nie[8] == LETRAS_NIF[numero % 23]

def validar_cif(cif):
    """CIF de empresa: letra + 7 dígitos + dígito/letra de control.

    Algoritmo AEAT:
    - Índices 0,2,4,6 del cuerpo (posiciones pares del CIF completo) → ×2
    - Índices 1,3,5 (posiciones impares) → suma directa
    """
    letra_tipo = cif[0]
    digitos    = cif[1:8]
    control    = cif[8]

    suma_pares = 0
    for i in [0, 2, 4, 6]:
        doble = int(digitos[i]) * 2
        suma_pares += doble if doble < 10 else doble - 9

    suma_impares  = sum(int(digitos[i]) for i in [1, 3, 5])
    total         = suma_pares + suma_impares
    control_num   = (10 - (total % 10)) % 10
    control_letra = "JABCDEFGHI"[control_num]

    if letra_tipo in "KPQS":
        return control == control_letra
    elif letra_tipo in "ABEH":
        return control == str(control_num)
    else:
        return control in (str(control_num), control_letra)

PATRON_NIF = re.compile(r"^\d{8}[A-Z]$")
PATRON_NIE = re.compile(r"^[XYZ]\d{7}[A-Z]$")
PATRON_CIF = re.compile(r"^[ABCDEFGHJKLMNPQRSUVW]\d{7}[0-9A-J]$")

def diagnosticar_nif(nif_raw):
    """Devuelve (es_valido, motivo) para cualquier identificador fiscal."""
    if pd.isna(nif_raw) or not isinstance(nif_raw, str):
        return False, "nulo"

    nif = nif_raw.strip().upper()

    # Algunos identificadores llevan el prefijo "ES" del NIF-IVA europeo — lo ignoramos
    if nif.startswith("ES") and len(nif) > 2:
        nif = nif[2:]

    if PATRON_NIF.match(nif):
        return (True, "ok") if validar_nif(nif) else (False, "checksum_incorrecto")
    if PATRON_NIE.match(nif):
        return (True, "ok") if validar_nie(nif) else (False, "checksum_incorrecto")
    if PATRON_CIF.match(nif):
        return (True, "ok") if validar_cif(nif) else (False, "checksum_incorrecto")

    # Diagnóstico de formato incorrecto
    if len(nif) < 9:
        return False, "longitud_corta"
    if len(nif) > 9:
        return False, "longitud_larga"
    return False, "formato_desconocido"

resultado = df["nif_contratista"].apply(diagnosticar_nif)
df["nif_valido"]        = resultado.apply(lambda x: x[0])
df["motivo_invalido"]   = resultado.apply(lambda x: x[1] if not x[0] else pd.NA)

invalidos = (~df["nif_valido"]).sum()
print(f"NIFs/CIFs con problemas: {invalidos} ({invalidos / len(df) * 100:.1f}%)")
print("\nDesglose por motivo:")
print(df["motivo_invalido"].value_counts())

NIFs/CIFs con problemas: 16 (2.5%)

Desglose por motivo:
motivo_invalido
checksum_incorrecto    6
longitud_corta         6
longitud_larga         3
formato_desconocido    1
Name: count, dtype: int64


In [22]:
# Listado de NIFs/CIFs inválidos con el motivo, ordenados por importe descendente
nif_invalidos = df.loc[
    ~df["nif_valido"],
    ["num_referencia", "nif_contratista", "nombre_contratista", "motivo_invalido", "presupuesto_base_eur"]
].sort_values("presupuesto_base_eur", ascending=False)

print(f"Total NIF/CIF inválidos: {len(nif_invalidos)}")
nif_invalidos

Total NIF/CIF inválidos: 16


,num_referencia,nif_contratista,nombre_contratista,motivo_invalido,presupuesto_base_eur
439,PRP2023/3242/3,B9416261,"PAZO DE RUBIANES, S.L",longitud_corta,"562,00"
566,PRP2023/4225/2,B3694069,TROULA ANIMACIN SL,longitud_corta,"5.150,00"
499,PRP2023/3880/5,G360018513,CINE CLUB ADEGA,longitud_larga,"484,00"
464,PRP2023/3669,A80364242,CLECE S.A,checksum_incorrecto,"437,94"
574,PRP2023/4292,B369400690,TROULA ANIMACIN SL,longitud_larga,"4.500,00"
405,PRP2023/2843,B9848241,"PIXELING,S.L.",longitud_corta,"3.940,00"
97,PRP2023/0062,B36299676,"DISTRIBUCIONES Y BEBIDAS POUSADA, SL",checksum_incorrecto,"3.300,00"
301,PRP/20231485/2,5248378Q,MARA CRUZ LPEZ MARTNEZ,longitud_corta,"272,00"
512,PRP2023/3870/9,B368992016,"VISUAL GRAPHICS IMAGEN Y PUBLICIDAD, S.L",longitud_larga,"203,85"
300,PRP/20231485/1,5248378Q,MARA CRUZ LPEZ MARTNEZ,longitud_corta,"2.272,72"


## 5 — Convertir importes de string español a float

Formato español: `"1.234,56"` → `1234.56`  
Proceso: quitar puntos de miles, sustituir coma decimal por punto, convertir a float.

In [23]:
def euros_a_float(valor):
    """Convierte un string de importe en formato español a float."""
    if pd.isna(valor) or str(valor).strip() in ("", "nan"):
        return pd.NA
    # Eliminamos el punto de miles y sustituimos la coma decimal por punto
    # Ejemplo: "1.234,56" → "1234.56"
    limpio = str(valor).strip().replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        # Si el valor contiene texto inesperado devolvemos nulo en lugar de fallar
        return pd.NA

COLS_IMPORTE = ["valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur"]

# Float64 con mayúscula es el tipo nullable de pandas — admite NaN, a diferencia de float64 de numpy
for col in COLS_IMPORTE:
    df[col] = df[col].apply(euros_a_float).astype("Float64")

print("Tipos resultantes:")
print(df[COLS_IMPORTE].dtypes)
print("\nEjemplos:")
df[COLS_IMPORTE].head()

Tipos resultantes:
valor_estimado_eur      Float64
presupuesto_base_eur    Float64
importe_sin_iva_eur     Float64
importe_con_iva_eur     Float64
dtype: object

Ejemplos:


,valor_estimado_eur,presupuesto_base_eur,importe_sin_iva_eur,importe_con_iva_eur
0,206356.56,206356.56,123401.22,149315.48
1,2394.0,2394.0,2394.0,2896.74
2,14950.0,14849.0,14849.0,17967.29
3,5559.96,5559.96,5559.96,6727.56
4,<NA>,39990.2,39852.22,48221.19


## 6 — Extraer el año del número de referencia

`Fecha formalización` tiene un 96% de nulos. El **número de referencia** incluye siempre el año,
pero en formatos distintos según el expediente:

- `CT 80/2022-e` → año después de `/`
- `EXP 2022/14` → año antes de `/`

La solución es buscar cualquier grupo de 4 dígitos en la referencia que sea un año plausible (2015–2030).
Además se añade un número secuencial de contrato dentro de cada año y una fecha estimada para los registros sin fecha real.

In [24]:
def extraer_anio(referencia):
    """Extrae el año buscando cualquier grupo de 4 dígitos plausible como año (2015-2030)."""
    if pd.isna(referencia):
        return pd.NA
    # Buscamos todos los grupos de 4 dígitos en la referencia
    candidatos = re.findall(r'\d{4}', str(referencia))
    for c in candidatos:
        anio = int(c)
        # Filtramos solo los que son años plausibles para este dataset
        if 2015 <= anio <= 2030:
            return anio
    return pd.NA

# Int64 con mayúscula permite nulos; int64 estándar no admite NaN
df["anio_contrato"] = df["num_referencia"].apply(extraer_anio).astype("Int64")

print("Distribución por año:")
print(df["anio_contrato"].value_counts().sort_index())
print(f"\nReferencias sin año extraíble: {df['anio_contrato'].isna().sum()}")

Distribución por año:
anio_contrato
2018      1
2019      1
2021      7
2022     60
2023    562
Name: count, dtype: Int64

Referencias sin año extraíble: 0


## 7 — Normalizar `fecha_formalizacion`

Solo un 4% de registros tiene fecha. La convertimos a `datetime` cuando existe.

In [25]:
# dayfirst=True indica que el formato es DD-MM-AAAA (formato español)
# errors="coerce" convierte a NaT (nulo de fecha) los valores que no se puedan parsear
df["fecha_formalizacion"] = pd.to_datetime(df["fecha_formalizacion"], dayfirst=True, errors="coerce")

con_fecha = df["fecha_formalizacion"].notna().sum()
print(f"Registros con fecha_formalizacion: {con_fecha} ({con_fecha / len(df) * 100:.1f}%)")
df.loc[df["fecha_formalizacion"].notna(), ["num_referencia", "fecha_formalizacion"]].head()

Registros con fecha_formalizacion: 26 (4.1%)


,num_referencia,fecha_formalizacion
0,CT 80/2022-e,2023-01-09
5,CT 23/2021-e,2023-04-18
6,CT 68/2022-e,2023-03-09
8,CT 84/2022-e,2023-03-07
26,CT 113/2018-e,2023-06-06


In [26]:
# Número secuencial de contrato dentro de cada año (útil para ordenar y analizar volumen anual)
df["num_contrato_anio"] = (
    df.groupby("anio_contrato").cumcount() + 1
).astype("Int64")

# Fecha estimada: usamos la real cuando existe; si no, generamos una aleatoria dentro del año.
# Semilla fija (seed=42) para que el resultado sea siempre reproducible.
rng = np.random.default_rng(seed=42)

# Identificamos los registros sin fecha real
sin_fecha = df["fecha_formalizacion"].isna()

def fecha_aleatoria_en_anio(anio):
    """Devuelve una fecha aleatoria dentro del año indicado."""
    if pd.isna(anio):
        return pd.NaT
    inicio = pd.Timestamp(f"{int(anio)}-01-01")
    dias   = 366 if inicio.is_leap_year else 365
    return inicio + pd.Timedelta(days=int(rng.integers(0, dias)))

# Generamos fechas aleatorias solo para los registros sin fecha real
fechas_sinteticas = df.loc[sin_fecha, "anio_contrato"].apply(fecha_aleatoria_en_anio)

# Copiamos fecha_formalizacion (ya es datetime64 en este punto) y rellenamos los huecos
df["fecha_estimada"] = df["fecha_formalizacion"].copy()
df.loc[sin_fecha, "fecha_estimada"] = fechas_sinteticas

print(f"Registros con fecha real:     {df['fecha_formalizacion'].notna().sum()}")
print(f"Registros con fecha estimada: {df['fecha_estimada'].notna().sum()}")
print("\nDistribución por año (fecha_estimada):")
print(df["fecha_estimada"].dt.year.value_counts().sort_index())

Registros con fecha real:     26
Registros con fecha estimada: 631

Distribución por año (fecha_estimada):
fecha_estimada
2021      3
2022     54
2023    574
Name: count, dtype: int64


## 8 — Filtrar: quedarse solo con contratos menores

In [27]:
antes = len(df)
# Filtramos para quedarnos solo con los contratos menores
df = df[df["tipo_contratacion"].str.strip() == "Contrato Menor"].copy()

print(f"Registros eliminados (Contrato Mayor): {antes - len(df)}")
print(f"Registros restantes (Contrato Menor):  {len(df)}")

# Eliminamos columnas que tras el filtro no aportan información analítica:
#   - codigo_cpv:           100 % nulos tras el filtro
#   - entidad_contratante:  1 único valor ("Ayuntamiento Vilagarcía de Arousa")
#   - tipo_contratacion:    usada solo para filtrar; ahora siempre "Contrato Menor"
#   - tipo_procedimiento:   siempre "Contrato menor"
#   - sistema_contratacion: siempre "No aplica"
#   - tramitacion:          siempre "Ordinaria"
COLUMNAS_CONSTANTES = [
    "codigo_cpv",
    "entidad_contratante",
    "tipo_contratacion",
    "tipo_procedimiento",
    "sistema_contratacion",
    "tramitacion",
]
df = df.drop(columns=COLUMNAS_CONSTANTES)

print(f"\nColumnas eliminadas por ser constantes o vacías: {len(COLUMNAS_CONSTANTES)}")
print(f"Columnas restantes: {df.shape[1]}")

Registros eliminados (Contrato Mayor): 24
Registros restantes (Contrato Menor):  607

Columnas eliminadas por ser constantes o vacías: 6
Columnas restantes: 19


## 9 — Convertir `plazo_meses` a numérico

In [28]:
# errors="coerce" convierte a NaN los valores que no sean numéricos en lugar de fallar
df["plazo_meses"] = pd.to_numeric(df["plazo_meses"], errors="coerce").astype("Float64")

print(df["plazo_meses"].describe())

count       607.0
mean      2.07911
std      3.546014
min          0.03
25%          0.03
50%           0.7
75%           2.0
max          12.0
Name: plazo_meses, dtype: Float64


## 10 — Marcar contratos cercanos al límite legal

Los contratos menores tienen límites legales:  
- Servicios y suministros: **≤ 15.000 €**  
- Obras: **≤ 40.000 €**

Añadimos `flag_limite` para detectar contratos que superan o rozan esos umbrales.

In [29]:
LIMITE_SERVICIOS = 15_000   # límite legal para servicios y suministros (€)
LIMITE_OBRAS     = 40_000   # límite legal para obras (€)
UMBRAL_ALERTA    = 0.90     # marcamos alerta si el importe supera el 90% del límite

def flag_limite(row):
    importe = row["presupuesto_base_eur"]
    tipo    = str(row["tipo_contrato"]).strip()
    if pd.isna(importe):
        return "sin_importe"
    # Las obras tienen un límite distinto al resto de tipos de contrato
    limite = LIMITE_OBRAS if tipo == "Obras" else LIMITE_SERVICIOS
    if importe > limite:
        return "supera_limite"       # contrato que incumple la ley de contratos
    if importe >= limite * UMBRAL_ALERTA:
        return "cerca_del_limite"    # posible fraccionamiento artificial del contrato
    return "ok"

# axis=1 aplica la función fila a fila (no columna a columna)
df["flag_limite"] = df.apply(flag_limite, axis=1)

print("Distribución de flags:")
print(df["flag_limite"].value_counts())

print("\nContratos que superan el límite legal:")
df.loc[
    df["flag_limite"] == "supera_limite",
    ["num_referencia", "objeto_contrato", "tipo_contrato", "presupuesto_base_eur", "nombre_contratista"]
]

Distribución de flags:
flag_limite
ok                  574
cerca_del_limite     33
Name: count, dtype: int64

Contratos que superan el límite legal:


,num_referencia,objeto_contrato,tipo_contrato,presupuesto_base_eur,nombre_contratista


## 11 — Revisión final de calidad

In [30]:
# Tabla resumen con tipo de dato, nulos y valores únicos de cada columna del dataset final
resumen = pd.DataFrame({
    "tipo":    df.dtypes,
    "nulos":   df.isnull().sum(),
    "% nulos": (df.isnull().mean() * 100).round(1),
    "únicos":  df.nunique(),  # valores distintos — útil para detectar columnas constantes
})
print(resumen.to_string())

                                tipo  nulos  % nulos  únicos
num_referencia                   str      0      0.0     607
objeto_contrato                  str      0      0.0     589
valor_estimado_eur           Float64      3      0.5     484
presupuesto_base_eur         Float64      0      0.0     488
plazo_meses                  Float64      0      0.0      43
tipo_contrato                    str      0      0.0       4
fecha_formalizacion   datetime64[us]    604     99.5       3
importe_sin_iva_eur          Float64      0      0.0     487
importe_con_iva_eur          Float64      0      0.0     492
url_licitacion                   str      0      0.0     188
observaciones                    str     64     10.5       2
contratistas_raw                 str      0      0.0     466
nif_contratista                  str      0      0.0     334
nombre_contratista               str      0      0.0     455
nif_valido                      bool      0      0.0       2
motivo_invalido         

In [31]:
# Estadísticas descriptivas de las columnas de importe (mín, máx, media, percentiles...)
print("=== Estadísticas de importes ===")
df[COLS_IMPORTE].describe().round(2)

=== Estadísticas de importes ===


,valor_estimado_eur,presupuesto_base_eur,importe_sin_iva_eur,importe_con_iva_eur
count,604.0,607.0,607.0,607.0
mean,3649.65,3741.59,3793.64,4520.6
std,5781.34,6010.12,6552.77,7691.23
min,16.53,16.53,16.53,20.0
25%,495.65,495.87,495.87,545.05
50%,1456.14,1488.9,1453.6,1733.6
75%,4133.48,4187.85,4050.91,4849.88
max,39981.73,39990.2,71300.0,71300.0


## 12 — Ordenar columnas y guardar

In [32]:
# Reordenamos las columnas en grupos lógicos para facilitar la lectura en Snowflake y Power BI
ORDEN_COLUMNAS = [
    # Identificación
    "num_referencia",
    "anio_contrato",
    "num_contrato_anio",
    "fecha_formalizacion",
    "fecha_estimada",
    # Clasificación
    "tipo_contrato",
    # Descripción
    "objeto_contrato",
    "plazo_meses",
    # Importes
    "valor_estimado_eur",
    "presupuesto_base_eur",
    "importe_sin_iva_eur",
    "importe_con_iva_eur",
    "flag_limite",
    # Contratista
    "nif_contratista",
    "nif_valido",
    "motivo_invalido",
    "nombre_contratista",
    "contratistas_raw",
    # Extra
    "observaciones",
    "url_licitacion",
]

df = df[ORDEN_COLUMNAS]

# encoding="utf-8-sig" añade BOM para que Excel y Power BI lean tildes y ñ correctamente
df.to_csv(RUTA_PROCESADO, index=False, encoding="utf-8-sig")

print(f"Guardado en: {RUTA_PROCESADO}")
print(f"Filas: {len(df)}  |  Columnas: {len(df.columns)}")

Guardado en: ../data/processed/contratos_limpios.csv
Filas: 607  |  Columnas: 20


## Resumen del paso de transformación

| Acción | Resultado |
|---|---|
| Columnas eliminadas (vacías) | 2 (`Unnamed: 0`, `Lista de lotes`) |
| Columnas renombradas | 18 → snake_case |
| Columnas nuevas | 6 (`nif_contratista`, `nombre_contratista`, `nif_valido`, `motivo_invalido`, `anio_contrato`, `num_contrato_anio`, `fecha_estimada`, `flag_limite`) |
| Importes convertidos | 4 columnas → `Float64` |
| Fecha parseada | `fecha_formalizacion` → `datetime64` |
| Contratos mayores eliminados | registros con `Tipo de contratación = Contrato Mayor` |
| Columnas constantes eliminadas | 6 (`codigo_cpv`, `entidad_contratante`, `tipo_contratacion`, `tipo_procedimiento`, `sistema_contratacion`, `tramitacion`) |
| Fichero de salida | `data/processed/contratos_limpios.csv` |